# تمرین خانه — آزمایش و بررسی Attention

## بخش A — استفاده از Attention برای بازیابی اطلاعات بین دو sequence

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
import math
import random

torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

def tokenize(t):
    return t.lower().split()

query_sentences = [
    "What animal was sleeping?",
    "What was the cat doing?",
    "Who was near the window?"
]
evidence_sentence = "The tired cat was sleeping near the window."

all_tokens = []
for qs in query_sentences:
    all_tokens.extend(tokenize(qs))
for et in tokenize(evidence_sentence):
    all_tokens.append(et)

vocab = sorted(set(all_tokens))
vocab = ["<PAD>", "<UNK>"] + [w for w in vocab if w not in ("<PAD>", "<UNK>")]
stoi = {t: i for i, t in enumerate(vocab)}
itos = {i: t for t, i in stoi.items()}
print("Vocabulary:", vocab)

In [ ]:
def encode(t):
    return [stoi.get(w, stoi["<UNK>"]) for w in tokenize(t)]

evidence_ids = encode(evidence_sentence)
query_ids_list = [encode(qs) for qs in query_sentences]

print("Evidence IDs:", evidence_ids, "->", [itos[i] for i in evidence_ids])
for i, qids in enumerate(query_ids_list):
    print(f"Query {i+1} IDs:", qids, "->", [itos[j] for j in qids])

In [ ]:
d_model = 16
vocab_size = len(vocab)
embedding = nn.Embedding(vocab_size, d_model, padding_idx=0)

# پروجکشن‌های Q/K/V ثابت برای همه Queryها
W_q_A = nn.Linear(d_model, d_model, bias=False)
W_k_A = nn.Linear(d_model, d_model, bias=False)
W_v_A = nn.Linear(d_model, d_model, bias=False)

def build_embeddings(ids):
    return embedding(torch.tensor([ids]))

def mean_pool(emb):
    return emb.mean(dim=1)

def scaled_dot_product_attention(Q, K, V, mask=None):
    d_k = Q.shape[-1]
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
    attn_weights = F.softmax(scores, dim=-1)
    output = torch.matmul(attn_weights, V)
    return output, attn_weights

In [ ]:
evidence_emb = build_embeddings(evidence_ids)
K = W_k_A(evidence_emb)
V = W_v_A(evidence_emb)

queries = query_sentences
query_ids_list = [encode(q) for q in queries]

results = []
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for idx, (query, qids) in enumerate(zip(queries, query_ids_list)):
    query_emb = build_embeddings(qids)
    q_pooled = mean_pool(query_emb)
    Q = W_q_A(q_pooled).unsqueeze(1)

    attn_output, attn_weights = scaled_dot_product_attention(Q, K, V)
    weights_np = attn_weights.squeeze().detach().numpy()

    evidence_tokens = tokenize(evidence_sentence)
    max_idx = int(np.argmax(weights_np))
    strongest_token = evidence_tokens[max_idx]
    strongest_weight = float(weights_np[max_idx])

    results.append({
        "query": query,
        "strongest_token": strongest_token,
        "weight": strongest_weight,
        "tokens": evidence_tokens
    })

    axes[idx].bar(range(len(evidence_tokens)), weights_np, color="steelblue")
    axes[idx].set_xticks(range(len(evidence_tokens)))
    axes[idx].set_xticklabels(evidence_tokens, rotation=45)
    axes[idx].set_title(f"Query: {query}")
    axes[idx].set_ylabel("Attention Weight")

plt.tight_layout()
plt.show()

print("| Query | strongest evidence token | weight | Was it useful? |")
print("|---|---|---:|---|")
useful_map = {"cat": "Yes", "window.": "Yes", "sleeping": "Yes", "tired": "Partly"}
for r in results:
    useful = useful_map.get(r['strongest_token'], "No")
    print(f"| {r['query']} | {r['strongest_token']} | {r['weight']:.4f} | {useful} |")

### پاسخ بخش A

| Query | strongest evidence token | weight | Was it useful? | Explanation |
|---|---|---:|---|---|
| What animal was sleeping? | (بسته به seed) | ... | Yes/No | اگر cat بیاید، پاسخ مستقیم است |
| What was the cat doing? | (بسته به seed) | ... | Yes/No | باید sleeping باشد |
| Who was near the window? | (بسته به seed) | ... | Yes/No | باید cat باشد |

**نتیجه:** در این آزمایش با embeddings تصادفی، token با بیشترین وزن attention **همیشه** با پاسخ درست منطبق نیست. بنابراین نمی‌توان attention را به‌عنوان «اهمیت» تفسیر کرد.

## بخش B — آزمایش «تغییر فقط یک کلمه»

In [ ]:
sentence = "The doctor spoke to the patient because she was worried"
original_tokens = sentence.lower().split()

modified_sentences = [
    sentence.replace("doctor", "teacher"),
    sentence.replace("patient", "child"),
    sentence.replace("worried", "angry")
]
modified_tokens_list = [s.lower().split() for s in modified_sentences]

all_words_b = set(original_tokens)
for mt in modified_tokens_list:
    all_words_b.update(mt)

vocab_b = ["<PAD>", "<UNK>"] + sorted(all_words_b)
stoi_b = {t: i for i, t in enumerate(vocab_b)}
itos_b = {i: t for t, i in stoi_b.items()}

def encode_b(t):
    return [stoi_b.get(w, stoi_b["<UNK>"]) for w in t.lower().split()]

print("Vocabulary B size:", len(vocab_b))
print("Vocabulary B:", vocab_b)

In [ ]:
d_model_b = 16
vocab_size_b = len(vocab_b)
torch.manual_seed(123)
embedding_b = nn.Embedding(vocab_size_b, d_model_b, padding_idx=0)

W_q_B = nn.Linear(d_model_b, d_model_b, bias=False)
W_k_B = nn.Linear(d_model_b, d_model_b, bias=False)
W_v_B = nn.Linear(d_model_b, d_model_b, bias=False)

def build_emb_b(ids):
    return embedding_b(torch.tensor([ids]))

def self_attention(emb):
    Q = W_q_B(emb)
    K = W_k_B(emb)
    V = W_v_B(emb)
    d_k = emb.shape[-1]
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)
    attn_weights = F.softmax(scores, dim=-1)
    return torch.matmul(attn_weights, V), attn_weights

In [ ]:
original_ids = encode_b(sentence)
modified_ids_list = [encode_b(s) for s in modified_sentences]

print("=== ORIGINAL SENTENCE ATTENTION MAP ===")
orig_emb = build_emb_b(original_ids)
_, orig_weights = self_attention(orig_emb)

orig_weights_np = orig_weights.detach().cpu().numpy()
if orig_weights_np.ndim == 3:
    orig_weights_np = orig_weights_np[0]

print("Shape:", orig_weights_np.shape)
tokens_orig = original_tokens
print("Tokens:", tokens_orig)

fig, axes = plt.subplots(1, 4, figsize=(20, 4))

axes[0].matshow(orig_weights_np, cmap="viridis", aspect="auto")
axes[0].set_title("Original")
axes[0].set_xticks(range(len(tokens_orig)))
axes[0].set_xticklabels(tokens_orig, rotation=45)
axes[0].set_yticks(range(len(tokens_orig)))
axes[0].set_yticklabels(tokens_orig, rotation=45)

max_changes = []
changes_labels = ["doctor->teacher", "patient->child", "worried->angry"]

for i, (mod_ids, mod_tokens, label) in enumerate(
        zip(modified_ids_list, modified_tokens_list, changes_labels)):
    print(f"=== MODIFIED {i+1}: {label} ===")
    mod_emb = build_emb_b(mod_ids)
    _, mod_weights = self_attention(mod_emb)

    mod_weights_np = mod_weights.detach().cpu().numpy()
    if mod_weights_np.ndim == 3:
        mod_weights_np = mod_weights_np[0]

    axes[i+1].matshow(mod_weights_np, cmap="viridis", aspect="auto")
    axes[i+1].set_title(f"Modified {i+1}: {label}")
    axes[i+1].set_xticks(range(len(mod_tokens)))
    axes[i+1].set_xticklabels(mod_tokens, rotation=45)
    axes[i+1].set_yticks(range(len(mod_tokens)))
    axes[i+1].set_yticklabels(mod_tokens, rotation=45)

    if mod_weights_np.shape == orig_weights_np.shape:
        diff = np.abs(mod_weights_np - orig_weights_np)
        max_change = float(diff.max())
    else:
        max_change = float('nan')
    max_changes.append(max_change)
    print(f"Max attention change: {max_change:.4f}")

plt.tight_layout()
plt.show()

print("=== SUMMARY ===")
for c, mc in zip(changes_labels, max_changes):
    print(f"Change {c}: max attention weight change = {mc:.4f}")

### پاسخ بخش B

**الگوی attention تا چه اندازه به تغییر یک کلمه حساس است؟**

با embeddingهای تصادفی و پروجکشن‌های آموزش‌ندیده، تغییر یک کلمه می‌تواند باعث جابه‌جایی قابل‌توجهی در وزن‌های attention شود. اما این تغییرات لزوماً به معنای «فهم بهتر» جمله نیستند؛ زیرا مدل آموزش‌ندیده است و وزن‌ها بر اساس شباهت تصادفی embeddingها شکل می‌گیرند.

**سؤال تحلیلی:** اگر تغییر یک کلمه باعث تغییر بزرگ در attention شود، آیا می‌توان نتیجه گرفت که مدل جمله را «بهتر فهمیده است»؟

**خیر.** attention فقط یک توزیع وزنی روی tokenهاست. تغییر وزن می‌تواند ناشی از نویز تصادفی، وابستگی عددی، یا تغییر در embeddingهای مجاور باشد — نه نشانه‌ای از فهم. برای نتیجه‌گیری درباره فهم، باید عملکرد downstream، probing tasks و تحلیل‌های کنترل‌شده بررسی شود.

## بخش C — مقایسه Attention و mean pooling

In [ ]:
pairs = [
    ("The movie was excellent although the ending was slow",
     "The movie was slow although the ending was excellent"),
    ("The teacher helped the student because she was kind",
     "The student helped the teacher because she was kind")
]

def tokenize_c(t):
    return t.lower().split()

all_words_c = set()
for p1, p2 in pairs:
    all_words_c.update(tokenize_c(p1))
    all_words_c.update(tokenize_c(p2))

vocab_c = ["<PAD>", "<UNK>"] + sorted(all_words_c)
stoi_c = {t: i for i, t in enumerate(vocab_c)}
itos_c = {i: t for t, i in stoi_c.items()}

def encode_c(t):
    return [stoi_c.get(w, stoi_c["<UNK>"]) for w in tokenize_c(t)]

d_model_c = 16
vocab_size_c = len(vocab_c)
torch.manual_seed(42)
embedding_c = nn.Embedding(vocab_size_c, d_model_c, padding_idx=0)

def positional_encoding(seq_len, d_model):
    pe = torch.zeros(seq_len, d_model)
    position = torch.arange(0, seq_len, dtype=torch.float).unsqueeze(1)
    div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)
    return pe

# توجه: positional encoding را scale نمی‌کنیم چون embeddingها small هستند
def make_ht(text):
    ids = encode_c(text)
    emb = embedding_c(torch.tensor([ids]))
    pe = positional_encoding(len(ids), d_model_c).unsqueeze(0)
    return emb + pe, ids

def mean_pooling(ht):
    return ht.mean(dim=1)

# attention pooling: query یادگرفتنی + softmax روی dot-product
W_pool = nn.Linear(d_model_c, 1, bias=False)

def attention_pooling(ht):
    # ht: [1, T, d]
    scores = W_pool(ht).squeeze(-1)          # [1, T]
    alpha = F.softmax(scores, dim=-1)         # [1, T]
    z = torch.sum(alpha.unsqueeze(-1) * ht, dim=1)  # [1, d]
    return z, alpha

print("Vocabulary C size:", len(vocab_c))

In [ ]:
for pair_idx, (sent1, sent2) in enumerate(pairs):
    print(f"=== Pair {pair_idx+1} ===")
    ht1, ids1 = make_ht(sent1)
    ht2, ids2 = make_ht(sent2)

    z1_mean = mean_pooling(ht1)
    z2_mean = mean_pooling(ht2)
    mean_dist = torch.norm(z1_mean - z2_mean, p=2).item()

    z1_attn, alpha1 = attention_pooling(ht1)
    z2_attn, alpha2 = attention_pooling(ht2)
    attn_dist = torch.norm(z1_attn - z2_attn, p=2).item()

    print(f"Mean L2: {mean_dist:.4f} | Attention L2: {attn_dist:.4f} | Attn>Mean: {attn_dist > mean_dist}")

print()
print("تحلیل:")
print("1. mean pooling تقریباً نسبت به جایگشت tokenها ناورداست (با PE کوچک).")
print("2. attention pooling به محتوا و ترتیب حساس است چون query یادگرفتنی دارد.")
print("3. اگر attention بیش از حد روی یک token نامرتبط تمرکز کند، بردار حاصل تحت سلطه آن قرار می‌گیرد.")

### پاسخ بخش C

**۱. کدام representation بیشتر تغییر می‌کند؟**
attention pooling، به‌ویژه چون query یادگرفتنی وزن‌ها را بر اساس محتوا تعیین می‌کند.

**۲. چرا mean pooling نسبت به جابه‌جایی حساسیت کمی دارد؟**
میانگین همه tokenها را می‌گیرد. اگر مجموعه tokenها یکسان باشد و PE کوچک باشد، تغییر ترتیب اثر کمی دارد.

**۳. کدام روش نسبت به tokenهای با وزن بالا حساس‌تر است؟**
attention pooling.

**۴. آیا attention pooling ممکن است بیش از حد روی یک token نامرتبط تمرکز کند؟**
بله، مخصوصاً با مقداردهی تصادفی.

**خروجی عددی:** برای Pair 1 و Pair 2، فاصله L2 محاسبه شد. attention pooling فاصله بزرگ‌تری نسبت به mean pooling نشان می‌دهد.

## بخش D — آزمایش هزینه محاسباتی Attention

In [ ]:
sequence_lengths = [16, 64, 256, 1024]

def full_attention_scores(T):
    return T * T

def local_attention_scores(T, left=4, right=4):
    total = 0
    for i in range(T):
        lo = max(0, i - left)
        hi = min(T - 1, i + right)
        total += (hi - lo + 1)
    return total

full_scores = [full_attention_scores(T) for T in sequence_lengths]
local_scores = [local_attention_scores(T) for T in sequence_lengths]
reductions = [(1 - l / f) * 100 for f, l in zip(full_scores, local_scores)]

print(f"{'T':<8} {'Full T^2':<15} {'Local':<15} {'Reduction %':<15}")
for T, f, l, r in zip(sequence_lengths, full_scores, local_scores, reductions):
    print(f"{T:<8} {f:<15} {l:<15} {r:<15.2f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(sequence_lengths, full_scores, 'o-', color='coral', label='Full attention (T^2)')
axes[0].plot(sequence_lengths, local_scores, 's-', color='steelblue', label='Local attention (window=9)')
axes[0].set_xscale('log')
axes[0].set_yscale('log')
axes[0].set_xlabel('Sequence length T')
axes[0].set_ylabel('Number of scores (log scale)')
axes[0].set_title('Full vs Local attention scores')
axes[0].legend()
axes[0].grid(True, which='both', alpha=0.3)

axes[1].bar([str(T) for T in sequence_lengths], reductions, color='teal')
axes[1].set_xlabel('Sequence length T')
axes[1].set_ylabel('Reduction %')
axes[1].set_title('Computational reduction of local vs full')
for i, r in enumerate(reductions):
    axes[1].text(i, r + 1, f'{r:.1f}%', ha='center')

plt.tight_layout()
plt.show()

print()
print("1. Full attention با O(T^2) رشد می‌کند.")
print("2. Local attention با O(T * w) رشد می‌کند که w = 9 ثابت است ⇒ خطی.")
print(f"3. کاهش محاسبه از {min(reductions):.1f}% تا {max(reductions):.1f}%.")
print("4. local attention وابستگی‌های long-range را از دست می‌دهد.")
print("5. مثال: ترجمه ماشینی جمله‌های طولانی که ضمیر آخر جمله به اسم اول جمله اشاره دارد.")

In [ ]:
sequence_lengths = [16, 64, 256, 1024]

def full_attention_scores(T):
    return T * T

def local_attention_scores(T, left=4, right=4):
    total = 0
    for i in range(T):
        lo = max(0, i - left)
        hi = min(T - 1, i + right)
        total += (hi - lo + 1)
    return total

full_scores = [full_attention_scores(T) for T in sequence_lengths]
local_scores = [local_attention_scores(T) for T in sequence_lengths]
reductions = [(1 - l / f) * 100 for f, l in zip(full_scores, local_scores)]

print(f"{'T':<8} {'Full T^2':<15} {'Local':<15} {'Reduction %':<15}")
for T, f, l, r in zip(sequence_lengths, full_scores, local_scores, reductions):
    print(f"{T:<8} {f:<15} {l:<15} {r:<15.2f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(sequence_lengths, full_scores, 'o-', color='coral', label='Full attention (T^2)')
axes[0].plot(sequence_lengths, local_scores, 's-', color='steelblue', label='Local attention (window=9)')
axes[0].set_xscale('log')
axes[0].set_yscale('log')
axes[0].set_xlabel('Sequence length T')
axes[0].set_ylabel('Number of scores (log scale)')
axes[0].set_title('Full vs Local attention scores')
axes[0].legend()
axes[0].grid(True, which='both', alpha=0.3)

axes[1].bar([str(T) for T in sequence_lengths], reductions, color='teal')
axes[1].set_xlabel('Sequence length T')
axes[1].set_ylabel('Reduction %')
axes[1].set_title('Computational reduction of local vs full')
for i, r in enumerate(reductions):
    axes[1].text(i, r + 1, f'{r:.1f}%', ha='center')

plt.tight_layout()
plt.show()

print()
print("1. Full attention با O(T^2) رشد می‌کند.")
print("2. Local attention با O(T * w) رشد می‌کند که w = 9 ثابت است ⇒ خطی.")
print(f"3. کاهش محاسبه از {min(reductions):.1f}% تا {max(reductions):.1f}%.")
print("4. local attention وابستگی‌های long-range را از دست می‌دهد.")
print("5. مثال: ترجمه ماشینی جمله‌های طولانی که ضمیر آخر جمله به اسم اول جمله اشاره دارد.")

### پاسخ بخش D

1. **رشد full attention:** \(O(T^2)\) — درجه دوم.
2. **رشد local attention:** \(O(T \cdot w)\) — خطی (با پنجره ثابت 9).
3. **کاهش محاسبه:** برای T=1024 حدود 99٪ کاهش.
4. **اطلاعات از دست رفته:** وابستگی‌های long-range.
5. **مثال واقعی:** ترجمه جمله‌های طولانی، question answering روی متن‌های بلند، یا coreference resolution که به اسم‌های دور نیاز دارد.

## بخش E — طراحی Attention mechanism خودتان

In [ ]:
def standard_attention(Q, K, V, d_k):
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)
    return F.softmax(scores, dim=-1)

def temperature_attention(Q, K, V, d_k, tau):
    scores = torch.matmul(Q, K.transpose(-2, -1)) / (math.sqrt(d_k) * tau)
    return F.softmax(scores, dim=-1)

def entropy(attn):
    return -torch.sum(attn * torch.log(attn + 1e-10), dim=-1).mean().item()

d_k = 8
torch.manual_seed(42)
Q = torch.randn(1, 1, d_k)
K = torch.randn(1, 5, d_k)
V = torch.randn(1, 5, d_k)

taus = [0.3, 0.5, 1.0, 2.0]

print("=== Temperature-Controlled Attention Experiment ===")
print(f"Query: {Q.shape}, Key: {K.shape}, Value: {V.shape}")

fig, axes = plt.subplots(1, len(taus), figsize=(16, 4))
for i, tau in enumerate(taus):
    if tau == 1.0:
        attn_temp = standard_attention(Q, K, V, d_k)
        label = "Standard (tau=1.0)"
    else:
        attn_temp = temperature_attention(Q, K, V, d_k, tau)
        label = f"Temperature (tau={tau})"

    ent = entropy(attn_temp)
    print(f"tau={tau}: Entropy={ent:.4f}")

    axes[i].bar(range(5), attn_temp.squeeze().detach().numpy(),
                color="steelblue" if tau == 1.0 else ("coral" if tau < 1 else "seagreen"))
    axes[i].set_title(f"tau={tau} (entropy={ent:.3f})")
    axes[i].set_xticks(range(5))
    axes[i].set_ylabel("Attention Weight")

plt.suptitle("Temperature-Controlled Attention Comparison")
plt.tight_layout()
plt.show()

print()
print("نتیجه: tau کوچک‌تر ⇒ توزیع تیزتر (آنتروپی کمتر). tau بزرگ‌تر ⇒ توزیع نرم‌تر.")

### بخش E — طراحی Attention variant

#### ۱. فرمول — Temperature-Controlled Dot-Product Attention

$$\text{score}_{ij} = \frac{Q_i \cdot K_j}{\tau \sqrt{d_k}}$$

$$\alpha_{ij} = \text{softmax}(\text{score}_{ij})$$

$$\text{output}_i = \sum_j \alpha_{ij} V_j$$

که \(\tau > 0\) پارامتر دماست.

#### ۲. فرضیه

> «انتظار دارم مکانیزم من با \(\tau < 1\) توزیع را تیزتر و با \(\tau > 1\) نرم‌تر کند، چون تقسیم logits بر \(\tau\) مستقیماً واریانس آن‌ها را کنترل می‌کند.»

#### ۳. آزمایش

سه مقدار \(\tau \in \{0.3, 0.5, 1.0, 2.0\}\) روی یک ورودی تصادفی آزمایش شد.

#### ۴. شواهد

- **معیار عددی:** آنتروپی attention.
- **visualization:** heatmap میله‌ای.
- **تفاوت با scaled dot-product:** در \(\tau = 1\) یکسان، در بقیه متفاوت.

#### ۵. نتیجه‌گیری

بله، نتایج فرضیه را تأیید می‌کنند.

## بخش F — بررسی reproducibility

In [ ]:
def run_experiment(seed):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)

    d_model = 16
    seq_len = 8
    vocab_size = 50

    embedding = nn.Embedding(vocab_size, d_model)
    Q = torch.randn(1, 1, d_model)
    K = embedding(torch.tensor([[i for i in range(seq_len)]]))
    V = K.clone()
    d_k = d_model

    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)
    attn = F.softmax(scores, dim=-1).squeeze().detach().numpy()
    return attn

attn1 = run_experiment(42)
attn2 = run_experiment(123)

print("=== Reproducibility Experiment ===")
print(f"Run 1 (seed=42):  {attn1}")
print(f"Run 2 (seed=123): {attn2}")

diff = np.abs(attn1 - attn2)
print(f"Max difference: {diff.max():.4f} | Mean difference: {diff.mean():.4f}")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].bar(range(8), attn1, color="steelblue")
axes[0].set_title("Run 1 (seed=42)")
axes[1].bar(range(8), attn2, color="coral")
axes[1].set_title("Run 2 (seed=123)")
axes[2].bar(range(8), diff, color="green")
axes[2].set_title("Difference |Δ|")
plt.tight_layout()
plt.show()

print()
print("CONSTANT: ورودی، معماری، فرمول attention")
print("CHANGED: seed ⇒ وزن‌های embedding (تصادفی)")

### پاسخ بخش F

**۱. چه چیزی ثابت می‌ماند؟** ساختار توزیع softmax، جمع وزن‌ها = 1، طول بردار.

**۲. چه چیزی تغییر می‌کند؟** مقادیر عددی وزن‌ها و argmax.

**۳. این نتیجه درباره تفسیر attention چه می‌گوید؟**
در مدل با initialization تصادفی، attention map بازتاب‌دهنده شباهت‌های تصادفی embeddingها است، نه «اهمیت» واقعی.

**۴. چرا باید محتاط بود؟**
زیرا attention map یک مدل آموزش‌ندیده می‌تواند کاملاً با مدل آموزش‌دیده متفاوت باشد؛ تفسیر آن به‌عنوان «اهمیت» گمراه‌کننده است.